# 01 — Ingest

Fetch the raw source and land it in `data/raw/` **untouched**.

**Source:** Box Office Mojo — *Top Lifetime Adjusted Grosses* (domestic). The
`?adjust_gross_to=2022` query param makes the page return each film's
inflation-adjusted gross (in 2022 dollars) alongside its nominal gross,
estimated tickets sold, and release year. With that param the page is static
(no JS needed).

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from io import StringIO
from src.ingest import load_config, fetch_html

cfg = load_config('config.yaml')
src = cfg['sources']['bom_adjusted']
url = src['url']
print('Source URL:', url)

## Fetch and cache the raw HTML
Save the page to `data/raw/` so the raw source is preserved and re-runs are
offline.

In [ ]:
raw_path = Path(cfg['paths']['data_raw']) / 'bom_top_lifetime_adjusted_2022.html'
raw_path.parent.mkdir(parents=True, exist_ok=True)

if raw_path.exists():
    print('Using cached raw HTML:', raw_path)
    html = raw_path.read_text(encoding='utf-8')
else:
    html = fetch_html(url)
    raw_path.write_text(html, encoding='utf-8')
    print('Saved raw HTML ->', raw_path)

print(f'{len(html):,} bytes')

## Parse the chart table
The page has a single table: Rank, Title, Adj. Lifetime Gross, Lifetime Gross,
Est. Num Tickets, Year.

In [ ]:
raw = pd.read_html(StringIO(html))[0]
print(raw.shape)
raw.head(10)

## Also fetch the WORLDWIDE chart
The adjusted chart above is **domestic only** (U.S. & Canada). To compare
domestic vs. international we also pull Box Office Mojo's worldwide chart,
which splits each film into worldwide / domestic / foreign gross.

In [ ]:
ww_url = cfg['sources']['bom_worldwide']['url']
ww_path = Path(cfg['paths']['data_raw']) / 'bom_ww_top_lifetime.html'
if ww_path.exists():
    ww_html = ww_path.read_text(encoding='utf-8')
else:
    ww_html = fetch_html(ww_url)
    ww_path.write_text(ww_html, encoding='utf-8')
ww_raw = pd.read_html(StringIO(ww_html))[0]
print(ww_raw.shape)
ww_raw.head(5)

## Third source: film GENRE from the TMDB API

Box Office Mojo has **no genre column**, so genre is a *third data source* we
ingest here — from **TMDB** (The Movie Database) via its REST API.

**This step requires a free API key.** Get one at themoviedb.org (Settings →
API) and put it in the project's `.env` file (gitignored, never committed):

```
TMDB_API_KEY=your_key_here
```

For each film we call `GET /3/search/movie?query=<title>&year=<year>`, take the
first result, and map its `genre_ids` to names via `GET /3/genre/movie/list`.
The HTTP call itself lives in `src/ingest.py` → `tmdb_lookup_movie()` (per the
workspace norm that fetch logic lives in `src/`), and every response is **cached
as JSON in `data/raw/tmdb/`** so this raw data is preserved and re-runs are
offline. The next cell makes **one call explicitly** so the mechanics are visible.

In [ ]:
import requests
from src.ingest import load_env, tmdb_genre_map, tmdb_lookup_movie

load_env('.env')  # load TMDB_API_KEY from the gitignored .env
api_key = os.environ.get('TMDB_API_KEY')
assert api_key, 'TMDB_API_KEY missing from .env — see the note above'

# One raw TMDB call, spelled out, so you can see what the API returns:
demo = requests.get('https://api.themoviedb.org/3/search/movie',
                    params={'api_key': api_key, 'query': 'Avatar', 'year': 2009}, timeout=20)
top = demo.json()['results'][0]
print('HTTP', demo.status_code, '| result:', top['title'], top['release_date'])
print('raw genre_ids:', top['genre_ids'])
print('mapped genres:', [tmdb_genre_map(api_key)[i] for i in top['genre_ids']])

### Ingest genre for every film
Take the film list straight from the two raw box-office tables just fetched, and
look up each on TMDB (first run hits the API ~321 times, rate-limited; re-runs
read the JSON cache in `data/raw/tmdb/`). The raw genre records are saved to
`data/raw/tmdb_genres.parquet` — this is the ingested genre data that
`02-clean.ipynb` will load into DuckDB (no API calls happen in cleaning).

In [ ]:
# film list from the raw tables (title + year), deduped
adj_titles = raw[['Title', 'Year']].rename(columns={'Title':'title','Year':'release_year'})
ww_titles  = ww_raw[['Title', 'Year']].rename(columns={'Title':'title','Year':'release_year'})
titles = pd.concat([adj_titles, ww_titles]).drop_duplicates().reset_index(drop=True)
titles['release_year'] = titles['release_year'].astype(int)

recs = [tmdb_lookup_movie(r['title'], int(r['release_year']), api_key, cfg)
        for _, r in titles.iterrows()]
genre_raw = pd.DataFrame(recs)
genre_raw['genres_str'] = genre_raw['genres'].apply(lambda g: ', '.join(g) if g else None)
genre_raw.to_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_genres.parquet')
print(f'{genre_raw["matched"].sum()}/{len(genre_raw)} films matched on TMDB')
genre_raw[['title','year','primary_genre','genres_str']].head(10)

### Also ingest each film's COUNTRY OF ORIGIN (TMDB)

Box Office Mojo's "domestic" means **U.S. & Canada** — which only equals a
film's *home market* for U.S.-made films. For a Chinese film like *Ne Zha 2*,
its money earned in China is bucketed as "foreign," which would make it look
like a film that conquered the world when it really conquered its home market.

So we also pull each film's **origin country** from TMDB's movie-detail
endpoint (`GET /3/movie/{id}`, via `tmdb_movie_country()` in `src/ingest.py`,
cached in `data/raw/tmdb/`). The `is_us` flag lets the domestic-vs-international
analysis stay consistent (home vs abroad) by restricting to U.S.-made films.
Non-U.S. films are kept in the data for a separate 'foreign films in the U.S.
market' angle.

In [ ]:
from src.ingest import tmdb_movie_country

country_rows = []
for r in genre_raw.to_dict('records'):
    if r.get('tmdb_id'):
        c = tmdb_movie_country(int(r['tmdb_id']), api_key, cfg)
        country_rows.append({'tmdb_id': int(r['tmdb_id']),
            'origin_country': ', '.join(c['origin_country']) if c['origin_country'] else None,
            'is_us': c['is_us']})
country_raw = pd.DataFrame(country_rows).drop_duplicates('tmdb_id')
country_raw.to_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_countries.parquet')
print(f'{int(country_raw["is_us"].sum())} US-made / {len(country_raw)-int(country_raw["is_us"].sum())} non-US')
country_raw['origin_country'].value_counts().head(8)

---
**Next:** `02-clean.ipynb` loads all three raw sources into DuckDB and cleans
them.

Nothing here modified data — the raw HTML and the TMDB JSON in `data/raw/` are
the untouched ingested sources.